In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset

# Setup Device
device = torch.device('cpu') # Pakai CPU saja biar aman

# 1. LOAD DATA & SPLIT (Harus sama persis dengan saat training)
df = pd.read_csv('../DATASETUAP/data_bersih.csv')
df.dropna(inplace=True)
sentences = df['text_clean'].values
labels = df['label'].values

# Split 80:20 (random_state=42 menjamin datanya sama dengan yang dipelajari model)
X_train, X_test, y_train, y_test = train_test_split(sentences, labels, test_size=0.2, random_state=42)

print(f"Jumlah Data Test yang akan dinilai: {len(X_test)}")

# ==========================================
# EVALUASI 1: MODEL LSTM
# ==========================================
print("\n--- MENILAI MODEL LSTM ---")
# Load Resources
vocab = joblib.load('../models/vocab.pkl')
model_lstm = torch.load('../models/model_lstm.pth', map_location=device) # Load state dict? Wait, need Class def first.

# Definisi Ulang Class LSTM (Wajib ada)
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()
        self.output_size = output_size
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=drop_prob, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        batch_size = x.size(0)
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)
        lstm_out = lstm_out.contiguous().view(-1, self.hidden_dim)
        out = self.dropout(lstm_out)
        out = self.fc(out)
        sig_out = self.sigmoid(out)
        sig_out = sig_out.view(batch_size, -1)
        sig_out = sig_out[:, -1]
        return sig_out, hidden

    def init_hidden(self, batch_size):
        weight = next(self.parameters()).data
        hidden = (weight.new(self.n_layers, batch_size, self.hidden_dim).zero_().to(device),
                  weight.new(self.n_layers, batch_size, self.hidden_dim).zero_().to(device))
        return hidden

# Init Model
vocab_size = len(vocab) + 1
model_lstm_struct = SentimentLSTM(vocab_size, 1, 200, 128, 2)
model_lstm_struct.load_state_dict(torch.load('../models/model_lstm.pth', map_location=device))
model_lstm_struct.eval()

# Prediksi LSTM
y_pred_lstm = []
seq_length = 100
for text in X_test:
    words = str(text).split()
    encoded = [vocab.get(w, 0) for w in words]
    features = np.zeros((1, seq_length), dtype=int)
    features[0, -len(encoded):] = np.array(encoded)[:seq_length]
    inputs = torch.from_numpy(features).to(device)
    h = model_lstm_struct.init_hidden(1)
    out, _ = model_lstm_struct(inputs, h)
    y_pred_lstm.append(1 if out.item() > 0.5 else 0)

acc_lstm = accuracy_score(y_test, y_pred_lstm)
print(f"Akurasi LSTM: {acc_lstm:.4f}")

# ==========================================
# EVALUASI 2: INDOBERT (Ambil sampel 100 data saja biar cepat)
# ==========================================
print("\n--- MENILAI MODEL INDOBERT (Sampel 100 data) ---")
# Kita ambil sampel karena CPU lama kalau prediksi 2000 data
X_test_sample = X_test[:100]
y_test_sample = y_test[:100]

tokenizer_bert = BertTokenizer.from_pretrained('../models/model_bert/')
model_bert = BertForSequenceClassification.from_pretrained('../models/model_bert/')
model_bert.to(device)
model_bert.eval()

y_pred_bert = []
for text in X_test_sample:
    inputs = tokenizer_bert(str(text), return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)
    with torch.no_grad():
        outputs = model_bert(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    y_pred_bert.append(pred)

acc_bert = accuracy_score(y_test_sample, y_pred_bert)
print(f"Akurasi IndoBERT: {acc_bert:.4f}")

# ==========================================
# EVALUASI 3: DISTILBERT (Sampel 100 data)
# ==========================================
print("\n--- MENILAI MODEL DISTILBERT (Sampel 100 data) ---")
tokenizer_distil = DistilBertTokenizer.from_pretrained('../models/model_distilbert/')
model_distil = DistilBertForSequenceClassification.from_pretrained('../models/model_distilbert/')
model_distil.to(device)
model_distil.eval()

y_pred_distil = []
for text in X_test_sample:
    inputs = tokenizer_distil(str(text), return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)
    with torch.no_grad():
        outputs = model_distil(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    y_pred_distil.append(pred)

acc_distil = accuracy_score(y_test_sample, y_pred_distil)
print(f"Akurasi DistilBERT: {acc_distil:.4f}")

# REKAP HASIL
print("\n============================================")
print("HASIL AKHIR UNTUK TABEL LAPORAN")
print("============================================")
print(f"1. LSTM Base Model   : {acc_lstm*100:.2f}%")
print(f"2. IndoBERT          : {acc_bert*100:.2f}%")
print(f"3. DistilBERT        : {acc_distil*100:.2f}%")

d:\SMESTER7\Machine Learning C\UAP\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Jumlah Data Test yang akan dinilai: 2634

--- MENILAI MODEL LSTM ---
Akurasi LSTM: 0.8424

--- MENILAI MODEL INDOBERT (Sampel 100 data) ---
Akurasi IndoBERT: 0.9800

--- MENILAI MODEL DISTILBERT (Sampel 100 data) ---
Akurasi DistilBERT: 0.9800

HASIL AKHIR UNTUK TABEL LAPORAN
1. LSTM Base Model   : 84.24%
2. IndoBERT          : 98.00%
3. DistilBERT        : 98.00%
